# Classificação do Fashion-MNIST usando uma Rede Neural Densa

Este notebook implementa uma solução de classificação de imagens utilizando uma Rede Neural com Arquitetura Totalmente Densa (Fully Connected / MLP) para reconhecer itens do dataset Fashion-MNIST.
O conjunto de dados contém 10 categorias de roupas e acessórios, cada imagem em escala de cinza com 28*28 pixels.

O objetivo é treinar um modelo capaz de identificar corretamente o tipo de vestuário presente em cada imagem, explorando uma arquitetura simples, porém eficaz, típica de modelos de Machine Learning em visão computacional.

**Origem**: https://keras.io/api/datasets/fashion_mnist/

**Etiquetas**:

<table>
<thead>
<tr>
<th style="text-align: center;">Label</th>
<th>Description</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">0</td>
<td>T-shirt/top</td>
</tr>
<tr>
<td style="text-align: center;">1</td>
<td>Trouser</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td>Pullover</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td>Dress</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td>Coat</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td>Sandal</td>
</tr>
<tr>
<td style="text-align: center;">6</td>
<td>Shirt</td>
</tr>
<tr>
<td style="text-align: center;">7</td>
<td>Sneaker</td>
</tr>
<tr>
<td style="text-align: center;">8</td>
<td>Bag</td>
</tr>
<tr>
<td style="text-align: center;">9</td>
<td>Ankle boot</td>
</tr>
</tbody>
</table>

## 0. Importação de Bibliotecas

In [1]:
import time
import numpy as np

from IPython.display import clear_output

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.datasets import fashion_mnist

from tensorflow.keras import mixed_precision

# Devido a warnings de "Allocation exceeds 10% of free system memory" vamos ativar o FP16
# RTX 3070 (Ampere) tem Tensor Cores que aceleram mixed precision (float16)
# Pesos -> Ficam em float32 (para estabilidade)
# Ativações e cálculos interno -> Em float16 nos Tensor Cores (mais rápido)
# Saída -> Também em float16
mixed_precision.set_global_policy("mixed_float16")

# Para reprodutibilidade dos experimentos
np.random.seed(42)

2025-11-16 15:09:21.809959: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Funções e callbacks

In [2]:
class PlotLearning(keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.losses = []
        self.val_losses = []
        self.acc = []
        self.val_acc = []
        self.epochs = []

    def on_epoch_end(self, epoch, logs={}):
        self.epochs.append(epoch + 1)
        self.losses.append(logs.get("loss"))
        self.val_losses.append(logs.get("val_loss"))
        self.acc.append(logs.get("accuracy"))
        self.val_acc.append(logs.get("val_accuracy"))

        clear_output(wait=True)

        fig = make_subplots(
            rows=1,
            cols=2,
            subplot_titles=("Loss", "Accuracy"),
            horizontal_spacing=0.12,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.losses,
                mode="lines+markers",
                name="train_loss",
                line=dict(color="#1f77b4"),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.val_losses,
                mode="lines+markers",
                name="val_loss",
                line=dict(color="#d62728"),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.acc,
                mode="lines+markers",
                name="train_acc",
                line=dict(color="#2ca02c"),
            ),
            row=1,
            col=2,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.val_acc,
                mode="lines+markers",
                name="val_acc",
                line=dict(color="#ff7f0e"),
            ),
            row=1,
            col=2,
        )

        # Axis labels
        fig.update_xaxes(title_text="Epoch", row=1, col=1)
        fig.update_xaxes(title_text="Epoch", row=1, col=2)
        fig.update_yaxes(title_text="Loss", rangemode="tozero", row=1, col=1)
        fig.update_yaxes(title_text="Accuracy", rangemode="tozero", row=1, col=2)

        fig.update_layout(
            height=350,
            width=880,
            showlegend=True,
            template="plotly_white",
            margin=dict(l=30, r=30, t=50, b=30),
        )

        fig.show()

## 2. Carregamento e Exploração dos Dados

### Sobre o Fashion MNIST
- Estrutura (28x28 pixels, grayscale)
- 10 classes de roupas: [T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot]
- 60,000 imagens de treino e 10,000 de validação

In [3]:
# Keras já fornece o Fashion MNIST preprocessado e dividido
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Mapeamento das labels para nomes legíveis
class_map = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot",
}

In [4]:
print(f"x_train.shape: {x_train.shape}")
print(f"y_train.shape: {y_train.shape}")
print(f"x_test.shape: {x_test.shape}")
print(f"y_test.shape: {y_test.shape}")
print(f"\nRange dos valores dos pixels: [{x_train.min()}, {x_train.max()}]")

x_train.shape: (60000, 28, 28)
y_train.shape: (60000,)
x_test.shape: (10000, 28, 28)
y_test.shape: (10000,)

Range dos valores dos pixels: [0, 255]


## 3. Visualização das Amostras

Visualizando os dados para entender o problema que estamos resolvendo

In [5]:
x_plot = x_train.astype("float32")

num_classes = len(np.unique(y_train))

fig = make_subplots(
    rows=1,
    cols=num_classes,
    subplot_titles=[class_map[i] for i in range(num_classes)],
    horizontal_spacing=0.01,
)

for idx in range(num_classes):
    # Pega a primeira imagem da classe idx
    image = x_plot[y_train == idx][0]

    # Corrige rotação (Fashion MNIST vem virado verticalmente)
    image = image[::-1, :]

    fig.add_trace(
        go.Heatmap(z=image, colorscale="gray", showscale=False),
        row=1,
        col=idx + 1,
    )

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(
    height=200, width=1000, showlegend=False, margin=dict(l=10, r=10, t=40, b=10)
)

fig.show()

## 4. Análise da Distribuição das Classes

Verificar se o dataset é balanceado

In [6]:
unique, counts = np.unique(y_train, return_counts=True)

fig = go.Figure(
    data=[
        go.Bar(
            x=[class_map[idx] for idx in unique],
            y=counts,
            marker=dict(color="steelblue"),
            opacity=0.85,
        )
    ]
)

fig.update_layout(
    title="Distribuição das Classes no Conjunto de Treino",
    xaxis_title="Classe",
    yaxis_title="Número de Amostras",
    width=500,
    height=320,
    margin=dict(l=40, r=20, t=60, b=80),
)

fig.update_xaxes(tickangle=30)

fig.show()

O dataset é balanceado!

## 5. Pré-processamento dos Dados

Normalização

Por que normalizar?
- Redes neurais funcionam melhor com valores pequenos (0-1) ao invés de (0-255)
- Acelera a convergência durante o treinamento
- Reduz problemas de gradientes explosivos/desaparecendo

Flatten vs Reshape
- Para redes densas: flatten para vetor 1D (784,)

In [7]:
# Normalização: escalar pixels de [0, 255] para [0, 1]
x_train_normalized = x_train.astype("float32") / 255.0
x_test_normalized = x_test.astype("float32") / 255.0

# Flatten: transformar imagens 28x28 em vetores de 784 elementos, o que acaba removendo a informação espacial!
x_train_flat = x_train_normalized.reshape(-1, 784)
x_test_flat = x_test_normalized.reshape(-1, 784)

print(f"x_train_flat: {x_train_flat.shape}")

assert (
    x_train_normalized[0][0][0] == x_train_flat[0][0]
), "Houve um erro, o primeiro pixel do 'x_train_normalized' e do 'x_train_flat' devem ser iguais!"

x_train_flat: (60000, 784)


### Codificação Categórica dos Labels

**One-Hot Encoding:** Converter labels de inteiros (0-9) para vetores binários (similar ao  [LabelBinarizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html) do scikit-learn)
- Exemplo: classe 2 -> [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
- Necessário para usar categorical_crossentropy como loss function
- Permite que o modelo retorne probabilidades para cada classe

In [8]:
num_classes = len(np.unique(y_train))
num_samples = y_train.shape[0]

y_train_binary = keras.utils.to_categorical(y_train, num_classes)
y_test_binary = keras.utils.to_categorical(y_test, num_classes)

print(
    f"Label original de '{class_map[y_train[0]]}': {y_train[0]}, binarizado: {y_train_binary[0]}"
)

assert y_train_binary.shape == (
    num_samples,
    num_classes,
), "Houve um erro, o número de linhas ou colunas é inconsistente"

Label original de 'Ankle boot': 9, binarizado: [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


## 6. Construção do Modelo - Arquitetura Densa

Trade-offs dessa arquitetura:
- Mais neurônios siginifica um maior capacidade de aprendizado, principalmente para dados mais complexos.
- Mas também é maior risco de `overfitting` e mais lento para treinar.


In [9]:
# Construção do modelo sequencial (camadas empilhadas linearmente)

model = Sequential(
    [
        # Input layer: 784 inputs (pixels)
        Input(shape=(784,), name="input_layer"),
        # Camada de entrada com 512 neurônios
        # ReLU: introduz não-linearidade e evita o `vanishing gradient`
        # Conecta TODOS os pixels a TODOS os neurônios (daí "densa")
        Dense(512, activation="relu", name="hidden_layer_1"),
        # Camada intermediária com 512 neurônios
        # Adiciona capacidade de aprender representações mais complexas
        # Permite que o modelo combine features da primeira camada
        Dense(512, activation="relu", name="hidden_layer_2"),
        # Camada de saída com 10 neurônios (correnpondendo as 10 classes da matriz binária)
        # 10 neurônios = 10 classes
        # Softmax: converte logits em probabilidades que somam 1
        # A classe com maior probabilidade é a predição final
        Dense(
            num_classes, activation="softmax", name="output_layer", dtype="float32"
        ),  # Converte a saída para fp32 novamente, para evitar problemas como Underflow e Overflow e falta de granularidade (FP16_problems.md)
    ],
    name="Fashion_MNIST_Dense_Model",
)

I0000 00:00:1763316566.259299  133171 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5562 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:06:00.0, compute capability: 8.6


## 7. Análise da Arquitetura do Modelo

### Calculando a quantidade de parâmetros:
- **Cálculo:** (784*512) + 512 + (512*512) + 512 + (512*10) + 10
- **Total de parâmetros:** ~670k parâmetros treináveis

In [10]:
model.summary()

Model: "Fashion_MNIST_Dense_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 669,706 (2.55 MB)

 Trainable params: 669,706 (2.55 MB)

 Non-trainable params: 0 (0.00 B)

## 8. Compilação do Modelo

### Componentes Essenciais:

**Loss Function (categorical_crossentropy):**
- Mede o quão "erradas" estão as predições
- Ideal para classificação multi-classe com one-hot encoding
- Fórmula: -Σ(y_true * log(y_pred))

**Optimizer (adam - implícito por padrão):**
- Adam = Adaptive Moment Estimation
- Ajusta a learning rate automaticamente
- Geralmente funciona bem sem tunning manual

**Metrics (accuracy):**
- % de predições corretas
- Fácil de interpretar e se cai bem já que nosso dataset é balanceado

In [11]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

## 9. Treinamento do Modelo

### Hiperparâmetros Importantes:

**Epochs (20):**
- Em cada época o modelo vê todos os dados de treino uma vez.
- Aumentar o número de épocas implica mais tempo de aprendizado.
- E também aumenta o risco de overfitting.

**Batch Size (32):**
- O batch size é a quantidade de amostras que a rede neural processa antes de realizar uma atualização nos pesos via backpropagation.
- Batches maiores   
Aproveitam melhor o paralelismo da GPU, permitindo treinar mais rápido por época.
Produzem um gradiente mais estável, porém podem levar a um modelo com menor capacidade de generalização.
- Batches menores
Geram mais passos de atualização por época, tornando o treinamento mais lento.
No entanto, o gradiente mais “ruidoso” pode atuar como uma forma de regularização, resultando às vezes em melhor desempenho final.
- Influenciado pela quantidade de VRAM/RAM

**Validation Data:**
- Usado para monitorar performance em dados não vistos.
- Quando observamos que a val_accuracy é muito menor que o train_accuracy temos um grande indício de `overfitting`.

Iniciando treinamento

In [12]:
plot_callback = PlotLearning()

train_start = time.time()

history = model.fit(
    x_train_flat,
    y_train_binary,
    epochs=20,
    batch_size=32,
    verbose=1,
    validation_data=(x_test_flat, y_test_binary),
    callbacks=[plot_callback],
)

train_end = time.time()

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9394 - loss: 0.1573 - val_accuracy: 0.8918 - val_loss: 0.4164


## 10. Análise dos Resultados de Treinamento
 
Pontos a serem obersvados:

- Training accuracy crescendo significa que o modelo está aprendendo
- Validation accuracy ficou estagnado, significa que apesar de performando bem no test, na validação não está, indicando um possível overfitting
- Quanto maior o `gap` entre train e val, mais overfitting
- O Validation loss aumentando é um sinal claro de overfitting

## 11. Avaliação Final no Conjunto de Teste

Esses resultados serão nosso `baseline`

In [13]:
test_loss, test_accuracy = model.evaluate(x_test_flat, y_test_binary, verbose=0)

inf_start = time.time()
model.predict(
    x_test_flat[[0]]
)  # Deveriamos rodar multiplas vezes para ter uma estimativa mais real
inf_end = time.time()

print(f"Total Training time: {train_end - train_start:.4f}s")
print(f"Inference time: {(inf_end - inf_start)*1000:.4f}ms/sample")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step
Total Training time: 113.6971s
Inference time: 256.5613ms/sample
Test Loss: 0.4164
Test Accuracy: 0.8918 (89.18%)


## 12. Matriz de Confusão - Análise Detalhada

A matriz de confusão mostra onde o modelo erra:
- Diagonal principal: predições corretas
- Off-diagonal: confusões entre classes
- Útil para identificar classes problemáticas

In [14]:
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

y_pred = model.predict(x_test_flat)
y_pred_classes = np.argmax(y_pred, axis=1)

# Gerar matriz de confusão
cm = confusion_matrix(y_test, y_pred_classes)

# Nomes das classes
labels = list(class_map.values())

# Criar heatmap
fig = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=labels,
        y=labels,
        colorscale="Blues",
        colorbar=dict(title="Número de Predições"),
        text=cm,
        texttemplate="%{text}",
        showscale=True,
    )
)

# Layout
fig.update_layout(
    title="Matriz de Confusão",
    xaxis=dict(title="Classe Predita", tickangle=30),
    yaxis=dict(title="Classe Real", autorange="reversed"),
    width=700,
    height=600,
    margin=dict(l=60, r=40, t=70, b=60),
)

fig.show()

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


## 13. Relatório de Classificação Detalhado

**Precision:** Das predições positivas, quantas estavam corretas?

**Recall:** Dos casos positivos reais, quantos foram detectados?

**F1-Score:** Média harmônica entre precision e recall

**Support:** Número de amostras de cada classe no teste

In [15]:
print(
    classification_report(
        y_test, y_pred_classes, target_names=class_map.values(), digits=4
    )
)

              precision    recall  f1-score   support

 T-shirt/top     0.8330    0.8380    0.8355      1000
     Trouser     0.9990    0.9630    0.9807      1000
    Pullover     0.7937    0.8500    0.8209      1000
       Dress     0.8724    0.9230    0.8970      1000
        Coat     0.8488    0.7690    0.8069      1000
      Sandal     0.9876    0.9540    0.9705      1000
       Shirt     0.7141    0.7070    0.7106      1000
     Sneaker     0.9308    0.9820    0.9557      1000
         Bag     0.9780    0.9790    0.9785      1000
  Ankle boot     0.9695    0.9530    0.9612      1000

    accuracy                         0.8918     10000
   macro avg     0.8927    0.8918    0.8917     10000
weighted avg     0.8927    0.8918    0.8917     10000



## 14. Visualização de Predições - Acertos e Erros

In [16]:
# Selecionar exemplos aleatórios
num_examples = 20

np.random.seed(42)
random_indices = np.random.choice(len(x_test), num_examples, replace=False)

fig = make_subplots(
    rows=4,
    cols=5,
    subplot_titles=[
        f"Real: {class_map[y_test[idx]]}<br>Pred: {class_map[y_pred_classes[idx]]}<br>({np.max(y_pred[idx])*100:.1f}%)"
        for idx in random_indices
    ],
    horizontal_spacing=0.02,
    vertical_spacing=0.08,
)

for i, idx in enumerate(random_indices):
    row = (i // 5) + 1
    col = (i % 5) + 1

    fig.add_trace(
        go.Heatmap(
            z=np.flipud(x_test[idx]),  # Inverte verticalmente
            colorscale="gray",
            showscale=False,
        ),
        row=row,
        col=col,
    )

    color = "green" if y_test[idx] == y_pred_classes[idx] else "red"
    fig.layout.annotations[i].update(font=dict(color=color, size=9))

fig.update_xaxes(showticklabels=False, matches=None)
fig.update_yaxes(showticklabels=False, matches=None, scaleanchor="x", scaleratio=1)
fig.update_layout(
    height=800,
    width=800,
    showlegend=False,
    title_text="Exemplos de Predições (Verde=Correto, Vermelho=Erro)",
    title_font_size=14,
    margin=dict(l=10, r=10, t=80, b=10),
)

fig.show()

## 15. Análise dos Erros Mais Confiantes

Erros com alta confiança indicam confusões sistemáticas no modelo

In [17]:
# Encontrar erros com alta confiança
errors_mask = y_test != y_pred_classes
confident_errors = np.where(errors_mask & (np.max(y_pred, axis=1) > 0.9))[0]

if len(confident_errors) > 0:
    print(f"Encontrados {len(confident_errors)} erros com confiança > 90%\n")

    # Mostrar até 10 exemplos
    num_to_show = min(10, len(confident_errors))

    fig = make_subplots(
        rows=2,
        cols=5,
        subplot_titles=[
            f"Real: {class_map[y_test[idx]]}<br>Pred: {class_map[y_pred_classes[idx]]}<br>({np.max(y_pred[idx])*100:.1f}%)"
            for idx in confident_errors[:num_to_show]
        ],
        horizontal_spacing=0.02,
        vertical_spacing=0.15,
    )

    for i, idx in enumerate(confident_errors[:num_to_show]):
        row = (i // 5) + 1
        col = (i % 5) + 1

        fig.add_trace(
            go.Heatmap(z=np.flipud(x_test[idx]), colorscale="gray", showscale=False),
            row=row,
            col=col,
        )

        fig.layout.annotations[i].update(font=dict(color="darkred", size=9))

    fig.update_xaxes(showticklabels=False, matches=None)
    fig.update_yaxes(showticklabels=False, matches=None, scaleanchor="x", scaleratio=1)
    fig.update_layout(
        height=400,
        width=800,
        showlegend=False,
        title_text="Erros Mais confiantes (>90% confiança)",
        title_font_size=14,
        margin=dict(l=10, r=10, t=80, b=10),
    )

    fig.show()
else:
    print("Nenhum erro com confiança > 90% encontrado!")

Encontrados 337 erros com confiança > 90%



## 16. Conclusões 

1. **Perda de Informação Espacial:** Flatten destrói relações entre pixels vizinhos
2. **Muitos Parâmetros:** ~670k parâmetros para imagens 28x28 é excessivo
3. **Overfitting:** Modelo tende a memorizar ao invés de generalizar
4. **Sem Invariância:** Não é robusto a translações, rotações ou escala

In [18]:
print(f"Arquitetura: Fully Connected (Dense)")
print(f"Tempo total de Treino: ~{(train_end-train_start):.2f} s")
print(f"Tempo de Inferência: ~{(inf_end-inf_start)*1000:.2f} ms/sample")
print(f"Parâmetros Treináveis: {model.count_params():,}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

Arquitetura: Fully Connected (Dense)
Tempo total de Treino: ~113.70 s
Tempo de Inferência: ~256.56 ms/sample
Parâmetros Treináveis: 669,706
Test Accuracy: 89.18%


In [19]:
# Descomente para salvar o modelo treinado
# model.save('fashion_mnist_dense_model.h5')